In [1]:
import pandas as pd
import numpy as np

In [55]:
df = pd.read_csv('IMDB.csv')
df.head()

,review,sentiment
0,Film version of Sandra Bernhard's one-woman of...,negative
1,I switched this on (from cable) on a whim and ...,positive
2,The `plot' of this film contains a few holes y...,negative
3,"Some amusing humor, some that falls flat, some...",negative
4,What can you say about this movie? It was not ...,negative


In [24]:
df.shape

(1000, 2)

In [25]:
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , f1_score , precision_score , recall_score
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import contractions
import emoji



In [13]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rayri\AppData\Roaming\nltk_data...


True

In [26]:
# data preprocessing


# normalize emoji's and enoticons
def normalize_emojis_and_emoticons(text):
    text = str(text)

    # Convert emojis to descriptive text
    text = emoji.demojize(text, delimiters=(" ", " "))

    # Normalize common emoticons
    emoticons = {
        r":\)": " smiley face ",
        r":-\)": " smiley face ",
        r":\(": " sad face ",
        r":-\(": " sad face ",
        r";\)": " wink face ",
        r":D": " laughing face ",
        r"<3": " heart "
    }

    for pattern, replacement in emoticons.items():
        text = re.sub(
            pattern,
            replacement,
            text,
            flags=re.IGNORECASE
        )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

# expand contraction words
def expand_contraction(text):
      return contractions.fix(str(text))

# handling missing and invalid reviews 
def handleMissingInvalid(df):
      df = df.dropna(subset=["review"])
      df["review"] = df["review"].astype(str)
      df = df[df["review"].str.strip().ne("")]
      return df

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    stop_words.difference_update({"no" , "not" , "nor" , "never" })
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
    return url_pattern.sub("", text)



In [27]:
def normalize_text(df):
    """Normalize review text for sentiment analysis."""
    try:
        df = handleMissingInvalid(df).copy()

        # Remove URLs before cleaning punctuation
        df["review"] = df["review"].apply(removing_urls)

        # Normalize sentiment-bearing symbols and contractions
        df["review"] = df["review"].apply(
            normalize_emojis_and_emoticons
        )
        df["review"] = df["review"].apply(expand_contraction)

        # Standardize text
        df["review"] = df["review"].apply(lower_case)
        df["review"] = df["review"].apply(removing_punctuations)
        df["review"] = df["review"].apply(removing_numbers)

        # Remove selected stop words and lemmatize
        df["review"] = df["review"].apply(remove_stop_words)
        df["review"] = df["review"].apply(lemmatization)

        return df

    except Exception as e:
        print(f"Error during text normalization: {e}")
        raise

In [28]:
test = df.head()
test

,review,sentiment
0,Film version of Sandra Bernhard's one-woman of...,negative
1,I switched this on (from cable) on a whim and ...,positive
2,The `plot' of this film contains a few holes y...,negative
3,"Some amusing humor, some that falls flat, some...",negative
4,What can you say about this movie? It was not ...,negative


In [29]:
result = normalize_text(test)
result

,review,sentiment
0,film version sandra bernhard one woman broadwa...,negative
1,switched cable whim treated quite surprise alt...,positive
2,plot film contains hole could drive massive tr...,negative
3,amusing humor fall flat decent acting quite at...,negative
4,say movie not terrible not good two day earlie...,negative


In [56]:
df = normalize_text(df)
df.head()

,review,sentiment
0,film version sandra bernhard one woman broadwa...,negative
1,switched cable whim treated quite surprise alt...,positive
2,plot film contains hole could drive massive tr...,negative
3,amusing humor fall flat decent acting quite at...,negative
4,say movie not terrible not good two day earlie...,negative


In [58]:
df['sentiment'].value_counts(dropna=False)

sentiment
0    517
1    483
Name: count, dtype: int64

In [57]:
df['sentiment'] = df['sentiment'].map({ 'positive':1 , 'negative':0 })
df.head()

,review,sentiment
0,film version sandra bernhard one woman broadwa...,0
1,switched cable whim treated quite surprise alt...,1
2,plot film contains hole could drive massive tr...,0
3,amusing humor fall flat decent acting quite at...,0
4,say movie not terrible not good two day earlie...,0


In [34]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [84]:
test_size = 0.25
n_features = 80
random_state = 42

In [85]:
X = df['review']  # Features
y = df['sentiment']  # Target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
X_train.shape , y_test.shape 
print(X_train.head())
print(y_train.head())


82     story line thief bagdad complex owing told fla...
991    ok dvd got new amazon mean think badass pretty...
789    temp not much theatrical business direct video...
894    mario fan long remember fond memory playing su...
398    like end day movie like b movie hoping would l...
Name: review, dtype: str
82     1
991    0
789    0
894    1
398    0
Name: sentiment, dtype: int64


In [86]:
vectorizer = CountVectorizer(max_features=n_features, stop_words='english')

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

print(X_train)
print(y_train)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 11336 stored elements and shape (750, 80)>
  Coords	Values
  (0, 65)	3
  (0, 34)	1
  (0, 51)	1
  (0, 79)	5
  (0, 41)	1
  (0, 47)	1
  (0, 74)	2
  (0, 39)	3
  (0, 11)	3
  (0, 69)	3
  (0, 71)	3
  (0, 55)	1
  (0, 45)	1
  (0, 40)	3
  (0, 26)	1
  (0, 70)	3
  (0, 46)	1
  (0, 72)	1
  (0, 76)	3
  (0, 14)	2
  (0, 44)	1
  (0, 48)	1
  (0, 49)	1
  (0, 24)	1
  (0, 58)	1
  :	:
  (748, 48)	1
  (748, 24)	3
  (748, 58)	1
  (748, 20)	5
  (748, 43)	3
  (748, 8)	2
  (748, 5)	1
  (748, 16)	3
  (748, 2)	1
  (748, 61)	1
  (748, 50)	1
  (748, 1)	1
  (748, 68)	1
  (749, 65)	2
  (749, 39)	2
  (749, 70)	1
  (749, 24)	1
  (749, 20)	1
  (749, 0)	1
  (749, 33)	2
  (749, 43)	4
  (749, 31)	1
  (749, 8)	4
  (749, 18)	1
  (749, 19)	1
82     1
991    0
789    0
894    1
398    0
      ..
106    0
270    0
860    0
435    0
102    1
Name: sentiment, Length: 750, dtype: int64


In [87]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [88]:
print(type(X_train))

<class 'numpy.ndarray'>


In [89]:
import dagshub


In [66]:
mlflow.set_tracking_uri(uri='https://dagshub.com/riteshray007/review_sentiment_analysis_endToEnd_mlops.mlflow')

dagshub.init(repo_owner='riteshray007',
             repo_name='review_sentiment_analysis_endToEnd_mlops',
             mlflow=True)

mlflow.set_experiment("LogisticRegression Baseline")



2026-10-11 18:20:29,126 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/riteshray007/review_sentiment_analysis_endToEnd_mlops "HTTP/1.1 200 OK"


Initialized MLflow to track repo "riteshray007/review_sentiment_analysis_endToEnd_mlops"

2026-10-11 18:20:29,135 - INFO - Initialized MLflow to track repo "riteshray007/review_sentiment_analysis_endToEnd_mlops"


Repository riteshray007/review_sentiment_analysis_endToEnd_mlops initialized!

2026-10-11 18:20:29,139 - INFO - Repository riteshray007/review_sentiment_analysis_endToEnd_mlops initialized!


<Experiment: artifact_location='mlflow-artifacts:/f35fae0de37e43ba96808356e22f9284', creation_time=1791721009434, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1791721009434, lifecycle_stage='active', name='LogisticRegression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [90]:
import time
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
      start_time = time.time()
      
      try:
            logging.info("logging preprocessing parameters ... ")
            mlflow.log_param("vectorizer" , "Bag of Words")
            mlflow.log_param("test size" , test_size)
            mlflow.log_param("num features ", n_features)
            
            logging.info("initializing logistic regression ")
            model = LogisticRegression(max_iter=1000)
            
            logging.info("fitting model ")
            model.fit(X_train,y_train)
            logging.info("model training complete")
            
            mlflow.log_param("model" , "logistic regression ")
            
            logging.info("making prediction ")
            y_pred = model.predict(X_test)
            
            logging.info("calculating evaluation metrics....")
            accuracy = accuracy_score( y_test , y_pred)
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            
            logging.info("Logging evaluation metrics...")
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            
            logging.info("Saving and logging the model...")
            mlflow.sklearn.log_model(model, "model")
            
            end_time = time.time()
            logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

            # Print the results for verification
            logging.info(f"Accuracy: {accuracy}")
            logging.info(f"Precision: {precision}")
            logging.info(f"Recall: {recall}")
            logging.info(f"F1 Score: {f1}")

            
      except Exception as e:
            logging.error(f"an error occurred - {e}" , exc_info=True)


2026-10-11 18:42:37,377 - INFO - Starting MLflow run...
2026-10-11 18:42:40,129 - INFO - logging preprocessing parameters ... 
2026-10-11 18:42:41,173 - INFO - initializing logistic regression 
2026-10-11 18:42:41,176 - INFO - fitting model 
2026-10-11 18:42:41,281 - INFO - model training complete
2026-10-11 18:42:41,705 - INFO - making prediction 
2026-10-11 18:42:41,713 - INFO - calculating evaluation metrics....
2026-10-11 18:42:41,781 - INFO - Logging evaluation metrics...
2026-10-11 18:42:43,167 - INFO - Saving and logging the model...
2026/10/11 18:42:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-10-11 18:43:08,741 - INFO - Model training and logging completed in 28.61 seconds.
2026-10-11 18:43:08,745 - INFO - Accuracy: 0.704
2026-10-11 18:43:08,750 - INFO - Precision: 0.7155172413793104
2026-10-11 18:43:08,753 - INFO - Recall: 0.6693548387096774
2026-10-11 18:43:08,756 - INFO - F1 Score: 0.6916666666666667


🏃 View run gentle-boar-635 at: https://dagshub.com/riteshray007/review_sentiment_analysis_endToEnd_mlops.mlflow/#/experiments/0/runs/8d100a8ceb4849bb953c7702357be4f2
🧪 View experiment at: https://dagshub.com/riteshray007/review_sentiment_analysis_endToEnd_mlops.mlflow/#/experiments/0
